# Lab Khai phá dữ liệu - Bài 2: Adult Income
**Trần Minh Tấn - 6451071071**

## Cell 1: Cài đặt và kết nối dữ liệu

In [ ]:
!pip install -q gdown

import gdown, pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns

HO_TEN = "Trần Minh Tấn"
MSSV   = "6451071071"

def info(cau=""):
    print("=" * 55)
    print(f"Họ tên : {HO_TEN}")
    print(f"MSSV   : {MSSV}")
    if cau:
        print(f"Nội dung: {cau}")
    print("=" * 55)

info("Cài đặt và kết nối dữ liệu")

TRAIN_ID = "1BYKP73pyzphA5YHR1fVvn2A0kBpr09TB"   # adult.data.csv
TEST_ID  = "1Vh7e1IBL0SRxjT3gU_v_rnIcAT18griD"   # adult.test.csv

gdown.download(id=TRAIN_ID, output="adult_data.csv", quiet=False)
gdown.download(id=TEST_ID,  output="adult_test.csv", quiet=False)

## Cell 2: Câu 1 - Nhập dữ liệu huấn luyện và kiểm thử

In [ ]:
info("Câu 1 - Nhập dữ liệu huấn luyện và kiểm thử")

cols = ["age","workclass","fnlwgt","education","education-num","marital-status",
        "occupation","relationship","race","sex","capital-gain","capital-loss",
        "hours-per-week","native-country","income"]

def doc_file(path):
    with open(path, encoding="utf-8-sig") as f:
        lines = f.readlines()
    skip = 1 if lines[0].lstrip().startswith("|") else 0                 # bỏ dòng thừa đầu file (nếu có)
    has_header = lines[skip].split(",")[0].strip().lower() == "age"      # file có dòng tên cột hay không
    d = pd.read_csv(path, skiprows=skip, header=0 if has_header else None, skipinitialspace=True)
    d.columns = cols if d.shape[1] == 15 else [c for c in cols if c != "fnlwgt"]
    d["income"] = d["income"].astype(str).str.strip().str.rstrip(".")    # bỏ dấu '.' cuối nhãn
    return d

train = doc_file("adult_data.csv")
test  = doc_file("adult_test.csv")

print("Số lượng dữ liệu huấn luyện:", len(train))
print("Số lượng dữ liệu kiểm thử  :", len(test))
print("Số cột tập huấn luyện:", train.shape[1], "| Số cột tập kiểm thử:", test.shape[1])

## Cell 3: Câu 2 - Tiền xử lý dữ liệu

In [ ]:
info("Câu 2 - Tiền xử lý dữ liệu")

train_n, test_n = len(train), len(test)

# Đánh dấu nguồn để tách lại ở câu 6
train["_set"] = "train"
test["_set"]  = "test"

# a) Xóa các dòng chứa dữ liệu trống '?'
train = train.replace("?", np.nan).dropna()
test  = test.replace("?", np.nan).dropna()

# b) Xóa cột fnlwgt
train = train.drop(columns=["fnlwgt"], errors="ignore")
test  = test.drop(columns=["fnlwgt"], errors="ignore")

# c) Nối train và test
df = pd.concat([train, test], ignore_index=True)

print(f"Train: {train_n} -> {(df['_set']=='train').sum()} dòng sau khi xóa dữ liệu trống")
print(f"Test : {test_n} -> {(df['_set']=='test').sum()} dòng sau khi xóa dữ liệu trống")
print("Kích thước dữ liệu sau khi nối:", df.shape)
df.head()

## Cell 4: Câu 3 - Độ tương đồng Pearson

In [ ]:
info("Câu 3 - Độ tương đồng Pearson")

num_cols = df.select_dtypes(include=np.number).columns
corr = df[num_cols].corr(method="pearson")
print(corr.round(3))

plt.figure(figsize=(7,5))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title(f"Pearson correlation\n{HO_TEN} - {MSSV}")
plt.show()

# Loại bỏ cột có độ tương đồng cao
THRESH = 0.8
to_drop = set()
for i in range(len(corr.columns)):
    for j in range(i):
        if abs(corr.iloc[i, j]) > THRESH:
            print(f"Tương quan cao: {corr.columns[i]} - {corr.columns[j]} = {corr.iloc[i,j]:.2f}")
            to_drop.add(corr.columns[i])

print("Cột bị loại:", list(to_drop) if to_drop else "Không có cột nào vượt ngưỡng " + str(THRESH))
df = df.drop(columns=list(to_drop))

## Cell 5: Câu 4 - Tách thuộc tính thường và thuộc tính quyết định

In [ ]:
info("Câu 4 - Tách thuộc tính thường và thuộc tính quyết định")

X = df.drop(columns=["income"])   # thuộc tính bình thường (còn cột _set để tách lại ở câu 6)
y = df["income"]                  # thuộc tính quyết định

print("Kích thước X:", X.shape)
print("Kích thước y:", y.shape)
print(y.value_counts())

## Cell 6: Câu 5 - Chuyển đổi cột không phải dạng số sang one-hot vector

In [ ]:
info("Câu 5 - Chuyển đổi cột không phải dạng số sang one-hot vector")

cat_cols = X.select_dtypes(include="object").columns.drop("_set")
print("Các cột dạng chữ:", list(cat_cols))

X_enc = pd.get_dummies(X, columns=cat_cols, dtype=int)
print("Số cột sau one-hot:", X_enc.shape[1])
X_enc.head()

## Cell 7: Câu 6 - Tách dữ liệu thành tập huấn luyện và kiểm thử như ban đầu

In [ ]:
info("Câu 6 - Tách dữ liệu thành tập huấn luyện và kiểm thử")

is_train = X_enc["_set"] == "train"
X_train = X_enc[is_train].drop(columns="_set")
X_test  = X_enc[~is_train].drop(columns="_set")
y_train = y[is_train]
y_test  = y[~is_train]

print("X_train:", X_train.shape, "| y_train:", y_train.shape)
print("X_test :", X_test.shape,  "| y_test :", y_test.shape)

## Cell 8: Câu 7 - Cây ID3

In [ ]:
info("Câu 7 - Cây ID3")

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (accuracy_score, classification_report,
                             ConfusionMatrixDisplay, precision_score,
                             recall_score, f1_score)

# ID3 dùng entropy (information gain)
id3 = DecisionTreeClassifier(criterion="entropy", max_depth=5, random_state=42)
id3.fit(X_train, y_train)
pred_id3 = id3.predict(X_test)

print("Accuracy ID3:", round(accuracy_score(y_test, pred_id3), 4))
print(classification_report(y_test, pred_id3))

# Ma trận nhầm lẫn
fig, ax = plt.subplots(figsize=(5,5))
ConfusionMatrixDisplay.from_predictions(y_test, pred_id3, cmap="Blues", ax=ax)
ax.set_title(f"Confusion matrix - ID3\n{HO_TEN} - {MSSV}")
plt.show()

# Biểu diễn cây
plt.figure(figsize=(24,10))
plot_tree(id3, feature_names=X_train.columns, class_names=id3.classes_,
          filled=True, fontsize=7, max_depth=3)
plt.title(f"Cây ID3 (hiển thị 3 mức đầu) - {HO_TEN} - {MSSV}")
plt.show()

## Cell 9: Câu 8 - Cây CART

In [ ]:
info("Câu 8 - Cây CART")

# CART dùng gini
cart = DecisionTreeClassifier(criterion="gini", max_depth=5, random_state=42)
cart.fit(X_train, y_train)
pred_cart = cart.predict(X_test)

print("Accuracy CART:", round(accuracy_score(y_test, pred_cart), 4))
print(classification_report(y_test, pred_cart))

fig, ax = plt.subplots(figsize=(5,5))
ConfusionMatrixDisplay.from_predictions(y_test, pred_cart, cmap="Greens", ax=ax)
ax.set_title(f"Confusion matrix - CART\n{HO_TEN} - {MSSV}")
plt.show()

plt.figure(figsize=(24,10))
plot_tree(cart, feature_names=X_train.columns, class_names=cart.classes_,
          filled=True, fontsize=7, max_depth=3)
plt.title(f"Cây CART (hiển thị 3 mức đầu) - {HO_TEN} - {MSSV}")
plt.show()

## Cell 10: Câu 9 - Naive Bayes

In [ ]:
info("Câu 9 - Naive Bayes")

from sklearn.naive_bayes import GaussianNB

nb = GaussianNB()
nb.fit(X_train, y_train)
pred_nb = nb.predict(X_test)

print("Accuracy Naive Bayes:", round(accuracy_score(y_test, pred_nb), 4))
print(classification_report(y_test, pred_nb))

fig, ax = plt.subplots(figsize=(5,5))
ConfusionMatrixDisplay.from_predictions(y_test, pred_nb, cmap="Oranges", ax=ax)
ax.set_title(f"Confusion matrix - Naive Bayes\n{HO_TEN} - {MSSV}")
plt.show()

## Cell 11: Câu 10 - So sánh kết quả các mô hình

In [ ]:
info("Câu 10 - So sánh kết quả các mô hình")

results = []
for name, pred in [("ID3", pred_id3), ("CART", pred_cart), ("Naive Bayes", pred_nb)]:
    results.append({
        "Mô hình":   name,
        "Accuracy":  accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, pos_label=">50K"),
        "Recall":    recall_score(y_test, pred, pos_label=">50K"),
        "F1-score":  f1_score(y_test, pred, pos_label=">50K"),
    })

res = pd.DataFrame(results).set_index("Mô hình").round(4)
display(res)

res.plot(kind="bar", figsize=(9,5), ylim=(0,1))
plt.title(f"So sánh các mô hình\n{HO_TEN} - {MSSV}")
plt.xticks(rotation=0)
plt.legend(loc="lower right")
plt.show()

best = res["Accuracy"].idxmax()
print(f"Mô hình có Accuracy cao nhất: {best} ({res.loc[best,'Accuracy']})")